In [ ]:
import pandas as pd
from preprocessing_utils import (
    RAW_DIR, latest_version, normalize_label, normalize_well_id,
    parse_date, parse_number, save_table,
)

concentration_df = pd.read_csv(RAW_DIR / "concentration.csv")
concentration_df["well_id"] = normalize_well_id(concentration_df["well_id"])
concentration_df["t"] = parse_date(concentration_df["t"])
for column in ["lab", "quality", "unit"]:
    concentration_df[column] = normalize_label(concentration_df[column])

concentration_df = latest_version(concentration_df, "sample_id", "result_version")
concentration_df = concentration_df.loc[
    concentration_df["quality"].isin(["ok", "replicate"])
].copy()

In [ ]:
unit_scale = concentration_df["unit"].map({"ug/l": 1, "mg/l": 1000})
report = concentration_df["concentration"].astype("string").str.strip().str.upper()
numeric_result = parse_number(report) * unit_scale

below_detection = {"<0.1": 0.08, "<0.05": 0.03, "ND": 0.01}
concentration_df["concentration"] = numeric_result.fillna(report.map(below_detection))
concentration_df["lod"] = parse_number(concentration_df["lod"]) * unit_scale
concentration_df = concentration_df.dropna(
    subset=["well_id", "t", "concentration", "lod"]
)
concentration_df = concentration_df.loc[concentration_df["concentration"].ge(0)].copy()
concentration_df["unit"] = "ug/l"

In [ ]:
display(concentration_df.head())
save_table(concentration_df, "concentration")
save_table(concentration_df, "concentration_df")